# Query Pipeline

A continuación, el código necesario para ejecutar el query pipeline según `instrucciones.md`

```
Código que:
(1) recibe la pregunta del usuario como entrada,
(2) convierte la pregunta a embedding,
(3) realiza una búsqueda vectorial (k-NN/ANN/por rango/híbrida),
(4) recupera los chunks relevantes,
(5) genera una respuesta con un LLM y
(6) devuelve un JSON con user_question, system_answer, chunks_related.
```

**Nota: Se debe indicar la misma variable de control aquí que la que se escogió en `00 data-pipeline.ipynb` para que agente consulte por la colección correspondiente.**

In [15]:
'''
Imports 
'''
from dotenv import load_dotenv
load_dotenv()
from os import getenv
from src.agent import Agent
from json import loads, dump
from pprint import pprint

print('Librerias importadas correctamente.')

'''
Variables de control
'''
# Descomentar solo una.
estrategia_chunking = [
    #'SEMANTICA',
    'FIXED_SIZE',
    # ...
]

if len(estrategia_chunking) != 1:
    raise Exception('Debe utilizarse solo una estrategia a la vez.')

estrategia_chunking = estrategia_chunking[0]
COLLECTION_NAME=str(getenv('COLLECTION_NAME'))

# Necesario para saber a que colección debe conectarse el agente
final_collection_name = f'{COLLECTION_NAME}_{estrategia_chunking}'

agente = Agent(final_collection_name)
# Cantidad de documentos en la db del agente
print(f'Cantidad de documentos en coleccion {final_collection_name}: {agente.db_collection.count()}')


Librerias importadas correctamente.
[2026-03-01 16:44:16 INFO] Agente inicializado exitosamente.
Cantidad de documentos en coleccion documentos_FIXED_SIZE: 36


## Ejecución del agente

La celda a continuación permite interactuar con el agente. Para ejecutar:

- Rellenar la variable `texto_para_consultar` con la consulta del usuario
- Ejecutar la celda.

Como resultado la celda imprimirá el JSON que devuelve el agente (con los campos `system_answer` y `chunks_related`) y adicionalmente imprimirá los fragmentos de texto que corresponden a los IDs mencionados en `chunks_related`.

Por temas de tiempo no incluyo un loop

In [16]:
# Consultar Agente
texto_para_consultar = 'Tengo una duda, que debo hacer en caso de accidente?'
respuesta = agente.onetime_query_model(texto_para_consultar)

# Imprimir respuesta del agente
print('#'*80)
pprint(object=respuesta)
print('#'*80)

# Imprimir fragmentos
fragmentos = agente.consultar_por_embedding_id(respuesta['chunks_related'])

for f in zip(list(respuesta['chunks_related']), fragmentos):
    print('-'*80)
    pprint(f)

[2026-03-01 16:44:20 INFO] Respuesta de API Embeddings recibida | Tiempo: 414 ms | Tokens - prompt: 16
[2026-03-01 16:44:20 INFO] Métricas escritas: ['2026-03-01T16:44:20', 16, 0, 16, '414', '3.2E-7USD']
[2026-03-01 16:44:20 INFO] Tokens estimados: 124
[2026-03-01 16:44:20 INFO] Consultando API... | Texto consulta: Tengo una duda, que debo hacer en caso de accidente? | Main Prompt: Eres del centro de enriquecimiento e investigación de Aperture Science. \nResponde la pregunta del usuario basándote UNICAMENTE en el contexto proporcionado.\nSi la informacion no esta en el contexto porporcionado a continuación,\ndi "No tengo informacion sobre eso en mi base de conocimientos"\n\nContexto:\n{contexto}\n\ncampos para responder:\n- system_answer: respuesta para el usuario basada en el contexto.\n- chunks_related: id del o los chunks que son relevantes para la respuesta.
[2026-03-01 16:44:22 INFO] Respuesta de API recibida | Tiempo: 1616 ms | Tokens - prompt: 528, respuesta: 82, total: 610 | Re

## Generacion de archivo de outputs.json

In [17]:
consultas = [
    'Tengo una duda, que debo hacer en caso de accidente?',
    'Holaa no tengo claro cuando pagan el sueldo aca',
    '¿Qué debo hacer en caso de terremoto? No veo señalética de evacuacion en las instalaciones.',
    'Nos invaden robots! Ayuda!',
]

respuestas = []

# Ejecutar consultas
for consulta in consultas:
    respuesta = agente.onetime_query_model(consulta)
    respuestas.append(respuesta)

# Escribir archivo
with open('./outputs/sample_queries_fixed.json', mode='a', encoding='utf-8') as file:
    dump(respuestas, file, ensure_ascii=False, indent=4)
    file.write("\n")  # opcional, para separar bloques

[2026-03-01 16:44:25 INFO] Respuesta de API Embeddings recibida | Tiempo: 243 ms | Tokens - prompt: 16
[2026-03-01 16:44:25 INFO] Métricas escritas: ['2026-03-01T16:44:25', 16, 0, 16, '243', '3.2E-7USD']
[2026-03-01 16:44:25 INFO] Tokens estimados: 124
[2026-03-01 16:44:25 INFO] Consultando API... | Texto consulta: Tengo una duda, que debo hacer en caso de accidente? | Main Prompt: Eres del centro de enriquecimiento e investigación de Aperture Science. \nResponde la pregunta del usuario basándote UNICAMENTE en el contexto proporcionado.\nSi la informacion no esta en el contexto porporcionado a continuación,\ndi "No tengo informacion sobre eso en mi base de conocimientos"\n\nContexto:\n{contexto}\n\ncampos para responder:\n- system_answer: respuesta para el usuario basada en el contexto.\n- chunks_related: id del o los chunks que son relevantes para la respuesta.
[2026-03-01 16:44:27 INFO] Respuesta de API recibida | Tiempo: 2189 ms | Tokens - prompt: 528, respuesta: 88, total: 616 | Re